# MEGA-CYBER LLM (MC-LLM) — Google Colab

Обучение **собственной** LLM с нуля прямо в Colab: свой токенизатор (CyberTokenizer, byte-level BPE), своя архитектура (decoder-only transformer), свои веса.

Что делает этот ноутбук:
1. Проверяет GPU.
2. Клонирует код.
3. Запускает полный pipeline: корпус → токенизатор → шарды → обучение → генерация.

> **В Colab открой `Runtime → Change runtime type → Hardware accelerator → GPU` (T4/A100).**

## 1. Проверить GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    print("BF16:", torch.cuda.is_bf16_supported())
else:
    print("!!! Нет GPU — включи его в Runtime > Change runtime type > GPU")


## 2. Получить код

**Вариант A** — клонировать из GitHub (замени URL на свой):

In [ ]:
# Замени на URL твоего репозитория
!git clone https://github.com/YOUR_USER/mega-cyber-llm.git
%cd mega-cyber-llm


**Вариант B** — если загрузил папку проекта в Colab (Files → Upload) или через Drive, просто перейди в неё:
```python
%cd /content/mega-cyber-llm
```

## 3. Установить зависимости и запустить обучение

In [ ]:
!pip install -q pyyaml tqdm

# Полный pipeline: корпус → токенизатор → шарды → 100M модель → обучение → генерация
# --steps 500 = 500 шагов оптимизатора (для быстрой проверки).
# Для более серьёзного обучения поставь --steps 5000 и подложи реальный корпус в data/raw/.
!python scripts/colab_train.py --steps 500


## 4. Сгенерировать текст из обученного чекпоинта

In [ ]:
!python scripts/generate.py --checkpoint checkpoints --prompt "Привет! Расскажи о себе." --max-tokens 60 --temperature 0.7
!python scripts/generate.py --checkpoint checkpoints --prompt "2 + 2 =" --max-tokens 10 --temperature 0


## 5. Интерактивный чат (опционально)

In [ ]:
# Один запрос через chat-шаблон
!python - <<'PY'
import torch
from inference.loader import load_model
from inference.generate import generate

model, tok, meta = load_model("checkpoints", "tokenizer")
device = "cuda" if torch.cuda.is_available() else "cpu"

while True:
    user = input("User: ")
    if user.strip().lower() in ("exit", "quit"):
        break
    prompt = tok.apply_chat_template([{"role": "user", "content": user}])
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    out = generate(model, ids, max_new_tokens=100, temperature=0.7, seed=None)
    print("Assistant:", tok.decode(out[0].tolist())[len(prompt):].strip())
PY


## Примечания

- **100M** (96M параметров) на T4 (16 GB) — легко, можно и 300M. На A100 — вплоть до 1B/3B.
- Демо-корпус синтетический (~12k строк). Для реального обучения положи свой корпус в `data/raw/` (`.txt`/`.jsonl`/`.md`) и убери `seed_corpus.jsonl`.
- Чекпоинты: `checkpoints/` (`.pt` + метаданные). Возобновить: `scripts/train.py --config configs/100m.yaml`.
- Для TPU (TRC) см. `docs/TRC_SETUP.md` и `training/tpu_backend.py`.
- 800B в Colab не обучить — нужен кластер. См. `scripts/estimate_800b.py`.
